In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

In [2]:
import numpy as np
def get_hop_distance(num_node, edge, max_hop=1):
    A = np.zeros((num_node, num_node))
    for i, j in edge:
        A[j, i] = 1
        A[i, j] = 1

    # compute hop steps
    hop_dis = np.zeros((num_node, num_node)) + np.inf
    transfer_mat = [np.linalg.matrix_power(A, d) for d in range(max_hop + 1)]
    arrive_mat = (np.stack(transfer_mat) > 0)
    for d in range(max_hop, -1, -1):
        hop_dis[arrive_mat[d]] = d
    return hop_dis


def normalize_digraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-1)
    AD = np.dot(A, Dn)
    return AD


def normalize_undigraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-0.5)
    DAD = np.dot(np.dot(Dn, A), Dn)
    return DAD
import numpy as np

class Graph():
    def __init__(self, strategy='spatial', max_hop=1, dilation=1):

        self.max_hop = max_hop
        self.dilation = dilation

        self.num_node = 25

        # ===== DEFINE EDGE =====
        self_link = [(i, i) for i in range(self.num_node)]

        neighbor_link = [
            (0, 1), (0, 15), (0, 16), (15, 17), (16, 18),
            (1, 2), (2, 3), (3, 4),
            (1, 5), (5, 6), (6, 7),
            (1, 8), (8, 9), (9, 10), (10, 11),
            (11, 24), (11, 22), (22, 23),
            (8, 12), (12, 13), (13, 14),
            (14, 21), (14, 19), (19, 20)
        ]

        self.edge = self_link + neighbor_link + [(j, i) for (i, j) in neighbor_link]

        # ===== CENTER =====
        self.center = 1

        # ===== CONNECT JOINT (CHO BONE) =====
        self.connect_joint = np.array([
            1, 1, 1, 2, 3,
            1, 5, 6,
            1, 8, 9, 10,
            8, 12, 13,
            0, 0, 15, 16,
            14, 19, 14,
            11, 22, 11
        ])

        # ===== HOP DIST =====
        self.hop_dis = get_hop_distance(
            self.num_node,
            self.edge,
            max_hop=max_hop
        )

        # ===== ADJ =====
        self.get_adjacency(strategy)

    def get_adjacency(self, strategy):

        valid_hop = range(0, self.max_hop + 1, self.dilation)

        adjacency = np.zeros((self.num_node, self.num_node))

        for hop in valid_hop:
            adjacency[self.hop_dis == hop] = 1

        normalize_adjacency = normalize_digraph(adjacency)

        if strategy == 'spatial':

            A = []

            for hop in valid_hop:
                a_root = np.zeros((self.num_node, self.num_node))
                a_close = np.zeros((self.num_node, self.num_node))
                a_far = np.zeros((self.num_node, self.num_node))

                for i in range(self.num_node):
                    for j in range(self.num_node):

                        if self.hop_dis[j, i] == hop:

                            if self.hop_dis[j, self.center] == self.hop_dis[i, self.center]:
                                a_root[j, i] = normalize_adjacency[j, i]

                            elif self.hop_dis[j, self.center] > self.hop_dis[i, self.center]:
                                a_close[j, i] = normalize_adjacency[j, i]

                            else:
                                a_far[j, i] = normalize_adjacency[j, i]

                if hop == 0:
                    A.append(a_root)
                else:
                    A.append(a_root + a_close)
                    A.append(a_far)

            A = np.stack(A)

        else:
            raise ValueError("Only spatial supported")

        self.A = A
import numpy as np
from torch.utils.data import Dataset


class SkeletonFeeder(Dataset):
    def __init__(self, data_path, label_path, graph, inputs='JVB', window=None):
        
        # load data
        self.data = np.load(data_path)   # (N, T, M, V, C)
        self.label = np.load(label_path) # (N,)

        # transpose → (N, C, T, V, M)
        self.data = self.data.transpose(0, 4, 1, 3, 2)
        self.data = self.data[:, :2, :, :, :]   # chỉ lấy x, y

        self.inputs = inputs
        self.conn = graph.connect_joint
        self.center = graph.center
        self.num_node = graph.num_node

        if window is not None:
            self.data = self.data[:, :, window[0]:window[1], :, :]

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        data = self.data[idx]  # (C, T, V, M)
        label = int(self.label[idx])

        joint, motion, bone = self.multi_input(data)

        out = []
        if 'J' in self.inputs:
            out.append(joint)
        if 'V' in self.inputs:
            out.append(motion)
        if 'B' in self.inputs:
            out.append(bone)

        out = np.stack(out, axis=0)  # (I, C, T, V, M)

        return out, label

    def multi_input(self, data):
        C, T, V, M = data.shape

        # Joint
        joint = data.copy()

        # Motion
        motion = np.zeros_like(data)
        motion[:, :-1] = data[:, 1:] - data[:, :-1]

        # Bone
        bone = np.zeros_like(data)
        for i in range(V):
            if i != self.conn[i]:
                bone[:, :, i, :] = data[:, :, i, :] - data[:, :, self.conn[i], :]

        return joint, motion, bone

In [3]:
class ConvTemporalGraphical(nn.Module):
    r"""The basic module for applying a graph convolution.

    Args:
        in_channels (int): Number of channels in the input sequence data
        out_channels (int): Number of channels produced by the convolution
        kernel_size (int): Size of the graph convolving kernel
        t_kernel_size (int): Size of the temporal convolving kernel
        t_stride (int, optional): Stride of the temporal convolution. Default: 1
        t_padding (int, optional): Temporal zero-padding added to both sides of
            the input. Default: 0
        t_dilation (int, optional): Spacing between temporal kernel elements.
            Default: 1
        bias (bool, optional): If ``True``, adds a learnable bias to the output.
            Default: ``True``

    Shape:
        - Input[0]: Input graph sequence in :math:`(N, in_channels, T_{in}, V)` format
        - Input[1]: Input graph adjacency matrix in :math:`(K, V, V)` format
        - Output[0]: Output graph sequence in :math:`(N, out_channels, T_{out}, V)` format
        - Output[1]: Graph adjacency matrix for output data in :math:`(K, V, V)` format

        where
            :math:`N` is a batch size,
            :math:`K` is the spatial kernel size, as :math:`K == kernel_size[1]`,
            :math:`T_{in}/T_{out}` is a length of input/output sequence,
            :math:`V` is the number of graph nodes. 
    """
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size,
                 t_kernel_size=1,
                 t_stride=1,
                 t_padding=0,
                 t_dilation=1,
                 bias=True):
        super().__init__()

        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels,
                              out_channels * kernel_size,
                              kernel_size=(t_kernel_size, 1),
                              padding=(t_padding, 0),
                              stride=(t_stride, 1),
                              dilation=(t_dilation, 1),
                              bias=bias)

    def forward(self, x, A):
        assert A.size(0) == self.kernel_size

        x = self.conv(x)

        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))

        return x.contiguous(), A


class Gconv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        if isinstance(kernel_size, int):
            gcn_kernel_size = kernel_size
            feature_dim = 0
        if isinstance(kernel_size, list) or isinstance(kernel_size, tuple):
            gcn_kernel_size = kernel_size[0]
            cnn_kernel_size = [1] + kernel_size[1:]
            feature_dim = len(kernel_size) - 1
        else:
            raise ValueError(
                'The type of kernel_size should be int, list or tuple.')

        if feature_dim == 1:
            self.conv = nn.Conv1d(in_channels,
                                  out_channels * gcn_kernel_size,
                                  kernel_size=cnn_kernel_size)
        elif feature_dim == 2:
            pass
        elif feature_dim == 3:
            pass
        elif feature_dim == 0:
            pass
        else:
            raise ValueError(
                'The length of kernel_size should be 1, 2, 3, or 4')

    def forward(self, X, A):
        pass
class asgcn(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 num_nodes, stride=1, dropout=0, residual=True):
        super().__init__()

        K = kernel_size[1]
        self.K = K

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, K)

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels,
                      (kernel_size[0], 1),
                      (stride, 1),
                      ((kernel_size[0] - 1) // 2, 0)),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True),
        )

        # ✅ ASGC core
        self.B = nn.Parameter(torch.zeros(K, num_nodes, num_nodes))
        nn.init.normal_(self.B, 0, 1e-3)

        self.WA = nn.Parameter(torch.ones(K, 1, 1))

        # residual
        if not residual:
            self.residual = lambda x: 0
        elif in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, (stride, 1)),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)

        A_hat = A * self.WA + self.B

        x, _ = self.gcn(x, A_hat)

        x = self.tcn(x) + res

        return self.relu(x), A_hat

In [4]:
import numpy as np
def get_hop_distance(num_node, edge, max_hop=1):
    A = np.zeros((num_node, num_node))
    for i, j in edge:
        A[j, i] = 1
        A[i, j] = 1

    # compute hop steps
    hop_dis = np.zeros((num_node, num_node)) + np.inf
    transfer_mat = [np.linalg.matrix_power(A, d) for d in range(max_hop + 1)]
    arrive_mat = (np.stack(transfer_mat) > 0)
    for d in range(max_hop, -1, -1):
        hop_dis[arrive_mat[d]] = d
    return hop_dis


def normalize_digraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-1)
    AD = np.dot(A, Dn)
    return AD


def normalize_undigraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-0.5)
    DAD = np.dot(np.dot(Dn, A), Dn)
    return DAD
import numpy as np

class Graph():
    def __init__(self, strategy='spatial', max_hop=1, dilation=1):

        self.max_hop = max_hop
        self.dilation = dilation

        self.num_node = 25

        # ===== DEFINE EDGE =====
        self_link = [(i, i) for i in range(self.num_node)]

        neighbor_link = [
            (0, 1), (0, 15), (0, 16), (15, 17), (16, 18),
            (1, 2), (2, 3), (3, 4),
            (1, 5), (5, 6), (6, 7),
            (1, 8), (8, 9), (9, 10), (10, 11),
            (11, 24), (11, 22), (22, 23),
            (8, 12), (12, 13), (13, 14),
            (14, 21), (14, 19), (19, 20)
        ]

        self.edge = self_link + neighbor_link + [(j, i) for (i, j) in neighbor_link]

        # ===== CENTER =====
        self.center = 1

        # ===== CONNECT JOINT (CHO BONE) =====
        self.connect_joint = np.array([
            1, 1, 1, 2, 3,
            1, 5, 6,
            1, 8, 9, 10,
            8, 12, 13,
            0, 0, 15, 16,
            14, 19, 14,
            11, 22, 11
        ])

        # ===== HOP DIST =====
        self.hop_dis = get_hop_distance(
            self.num_node,
            self.edge,
            max_hop=max_hop
        )

        # ===== ADJ =====
        self.get_adjacency(strategy)

    def get_adjacency(self, strategy):

        valid_hop = range(0, self.max_hop + 1, self.dilation)

        adjacency = np.zeros((self.num_node, self.num_node))

        for hop in valid_hop:
            adjacency[self.hop_dis == hop] = 1

        normalize_adjacency = normalize_digraph(adjacency)

        if strategy == 'spatial':

            A = []

            for hop in valid_hop:
                a_root = np.zeros((self.num_node, self.num_node))
                a_close = np.zeros((self.num_node, self.num_node))
                a_far = np.zeros((self.num_node, self.num_node))

                for i in range(self.num_node):
                    for j in range(self.num_node):

                        if self.hop_dis[j, i] == hop:

                            if self.hop_dis[j, self.center] == self.hop_dis[i, self.center]:
                                a_root[j, i] = normalize_adjacency[j, i]

                            elif self.hop_dis[j, self.center] > self.hop_dis[i, self.center]:
                                a_close[j, i] = normalize_adjacency[j, i]

                            else:
                                a_far[j, i] = normalize_adjacency[j, i]

                if hop == 0:
                    A.append(a_root)
                else:
                    A.append(a_root + a_close)
                    A.append(a_far)

            A = np.stack(A)

        else:
            raise ValueError("Only spatial supported")

        self.A = A
import numpy as np
from torch.utils.data import Dataset


class SkeletonFeeder(Dataset):
    def __init__(self, data_path, label_path, graph, inputs='JVB', window=None):
        
        # load data
        self.data = np.load(data_path)   # (N, T, M, V, C)
        self.label = np.load(label_path) # (N,)

        # transpose → (N, C, T, V, M)
        self.data = self.data.transpose(0, 4, 1, 3, 2)
        self.data = self.data[:, :2, :, :, :]   # chỉ lấy x, y

        self.inputs = inputs
        self.conn = graph.connect_joint
        self.center = graph.center
        self.num_node = graph.num_node

        if window is not None:
            self.data = self.data[:, :, window[0]:window[1], :, :]

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        data = self.data[idx]  # (C, T, V, M)
        label = int(self.label[idx])

        joint, motion, bone = self.multi_input(data)

        out = []
        if 'J' in self.inputs:
            out.append(joint)
        if 'V' in self.inputs:
            out.append(motion)
        if 'B' in self.inputs:
            out.append(bone)

        out = np.stack(out, axis=0)  # (I, C, T, V, M)

        return out, label

    def multi_input(self, data):
        C, T, V, M = data.shape

        # Joint
        joint = data.copy()

        # Motion
        motion = np.zeros_like(data)
        motion[:, :-1] = data[:, 1:] - data[:, :-1]

        # Bone
        bone = np.zeros_like(data)
        for i in range(V):
            if i != self.conn[i]:
                bone[:, :, i, :] = data[:, :, i, :] - data[:, :, self.conn[i], :]

        return joint, motion, bone

In [5]:
import torch
import torch.nn as nn

class ASGCN_Model(nn.Module):
    def __init__(self, num_class=7, num_point=25, num_person=9, in_channels=2):
        super().__init__()

        self.num_person = num_person

        # ===== 3 STREAMS =====
        self.stream_J = nn.Sequential(
            asgcn(in_channels, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 32, kernel_size=(9,3), num_nodes=num_point),
        )

        self.stream_V = nn.Sequential(
            asgcn(in_channels, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 32, kernel_size=(9,3), num_nodes=num_point),
        )

        self.stream_B = nn.Sequential(
            asgcn(in_channels, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 64, kernel_size=(9,3), num_nodes=num_point),
            asgcn(64, 32, kernel_size=(9,3), num_nodes=num_point),
        )

        # ===== FUSION =====
        self.layer1 = asgcn(96, 128, kernel_size=(9,3), num_nodes=num_point, stride=2)
        self.layer2 = asgcn(128, 128, kernel_size=(9,3), num_nodes=num_point)
        self.layer3 = asgcn(128, 128, kernel_size=(9,3), num_nodes=num_point)

        self.layer4 = asgcn(128, 256, kernel_size=(9,3), num_nodes=num_point, stride=2)
        self.layer5 = asgcn(256, 256, kernel_size=(9,3), num_nodes=num_point)
        self.layer6 = asgcn(256, 256, kernel_size=(9,3), num_nodes=num_point)

        self.bn = nn.BatchNorm2d(256)
        self.fc = nn.Linear(256, num_class)

    def forward(self, x, A):
        # x: (N, I=3, C=2, T, V, M)
        N, I, C, T, V, M = x.shape

        # ===== SPLIT 3 STREAM =====
        x_J = x[:, 0]   # (N, C, T, V, M)
        x_V = x[:, 1]
        x_B = x[:, 2]

        # ===== reshape (merge person) =====
        x_J = x_J.permute(0, 4, 1, 2, 3).contiguous().view(N*M, C, T, V)
        x_V = x_V.permute(0, 4, 1, 2, 3).contiguous().view(N*M, C, T, V)
        x_B = x_B.permute(0, 4, 1, 2, 3).contiguous().view(N*M, C, T, V)

        # ===== STREAM J =====
        for layer in self.stream_J:
            x_J, A = layer(x_J, A)

        # ===== STREAM V =====
        for layer in self.stream_V:
            x_V, A = layer(x_V, A)

        # ===== STREAM B =====
        for layer in self.stream_B:
            x_B, A = layer(x_B, A)

        # ===== CONCAT =====
        x = torch.cat([x_J, x_V, x_B], dim=1)  # (N*M, 96, T, V)

        # ===== FUSION =====
        x, A = self.layer1(x, A)
        x, A = self.layer2(x, A)
        x, A = self.layer3(x, A)

        x, A = self.layer4(x, A)
        x, A = self.layer5(x, A)
        x, A = self.layer6(x, A)

        x = self.bn(x)

        # ===== GLOBAL POOL =====
        x = x.mean(dim=[2,3])  # (N*M, 256)

        # ===== MERGE PERSON =====
        x = x.view(N, M, -1).mean(dim=1)

        return self.fc(x)

In [6]:
# =========================================================
# IMPORT
# =========================================================

import numpy as np
import torch

from torch.utils.data import Dataset

# =========================================================
# CONFIG
# =========================================================

TARGET_FRAMES = 16

# =========================================================
# SKELETON FEEDER
# =========================================================

class SkeletonFeeder(Dataset):

    def __init__(
        self,
        data_path,
        label_path,
        graph,
        inputs='JVB',
        window=None
    ):

        # =================================================
        # LOAD DATA
        # =================================================

        self.data = np.load(data_path)      # (N,T,M,V,C)
        self.label = np.load(label_path)    # (N,)

        self.inputs = inputs

        self.conn = graph.connect_joint
        self.center = graph.center
        self.num_node = graph.num_node

        # =================================================
        # TEMPORAL WINDOW
        # =================================================

        if window is not None:

            self.data = self.data[
                :,
                window[0]:window[1],
                :,
                :,
                :
            ]

    # =====================================================
    # SAMPLE 30 FRAMES
    # =====================================================

    def sample_skeleton(self, skel):

        """
        Input:
            skel shape = (T,M,V,C)

        Output:
            (30,M,V,C)
        """

        T = skel.shape[0]

        # =============================================
        # UNIFORM SAMPLING
        # =============================================

        if T >= TARGET_FRAMES:

            idx = np.linspace(
                0,
                T - 1,
                TARGET_FRAMES
            ).astype(int)

            skel = skel[idx]

        # =============================================
        # PAD IF SHORT
        # =============================================

        else:

            pad_shape = (
                TARGET_FRAMES - T,
                *skel.shape[1:]
            )

            pad = np.zeros(
                pad_shape,
                dtype=skel.dtype
            )

            skel = np.concatenate(
                [skel, pad],
                axis=0
            )

        return skel

    # =====================================================
    # MULTI INPUT
    # =====================================================

    def multi_input(self, data):

        """
        Input:
            data shape = (C,T,V,M)

        Output:
            joint  = (C,T,V,M)
            motion = (C,T,V,M)
            bone   = (C,T,V,M)
        """

        C, T, V, M = data.shape

        # =================================================
        # JOINT
        # =================================================

        joint = data.copy()

        # =================================================
        # MOTION
        # =================================================

        motion = np.zeros_like(data)

        motion[:, :-1] = (
            data[:, 1:] - data[:, :-1]
        )

        motion[:, -1] = motion[:, -2]

        # =================================================
        # BONE
        # =================================================

        bone = np.zeros_like(data)

        for i in range(V):

            if i != self.conn[i]:

                bone[:, :, i, :] = (
                    data[:, :, i, :]
                    -
                    data[:, :, self.conn[i], :]
                )

        return joint, motion, bone

    # =====================================================
    # BASIC
    # =====================================================

    def __len__(self):

        return len(self.label)

    # =====================================================
    # GET ITEM
    # =====================================================

    def __getitem__(self, idx):

        # =================================================
        # LABEL
        # =================================================

        label = int(self.label[idx])

        # =================================================
        # LOAD SKELETON
        # =================================================

        skel = self.data[idx]    # (T,M,V,C)

        # =================================================
        # SAMPLE FRAME
        # =================================================

        skel = self.sample_skeleton(skel)

        # =================================================
        # TRANSPOSE
        # (T,M,V,C) -> (C,T,V,M)
        # =================================================

        skel = skel.transpose(
            3, 0, 2, 1
        )

        # =================================================
        # ONLY X,Y
        # =================================================

        skel = skel[:2].astype(np.float32)

        # =================================================
        # NORMALIZE
        # =================================================

        skel = (
            skel - skel.mean()
        ) / (
            skel.std() + 1e-6
        )

        # =================================================
        # MULTI STREAM
        # =================================================

        joint, motion, bone = self.multi_input(skel)

        # =================================================
        # SELECT INPUT STREAM
        # =================================================

        out = []

        if 'J' in self.inputs:
            out.append(joint)

        if 'V' in self.inputs:
            out.append(motion)

        if 'B' in self.inputs:
            out.append(bone)

        # =================================================
        # STACK
        # (I,C,T,V,M)
        # =================================================

        out = np.stack(
            out,
            axis=0
        )

        # =================================================
        # TO TENSOR
        # =================================================

        out = torch.tensor(
            out,
            dtype=torch.float32
        )

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return out, label

In [ ]:
graph = Graph(strategy='spatial')

train_ds = SkeletonFeeder('../X_train.npy', '../y_train.npy', graph)
val_ds   = SkeletonFeeder('../X_val.npy', '../y_val.npy', graph)
test_ds  = SkeletonFeeder('../X_test.npy', '../y_test.npy', graph)
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)
test_loader  = DataLoader(test_ds, batch_size=64)
for x, y in train_loader:
    print(x.shape)
    print(y.shape)
    break

torch.Size([32, 3, 2, 16, 25, 9])
torch.Size([32])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# ===================== DEVICE =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ===================== GRAPH =====================
graph = Graph(strategy='spatial')
A = torch.tensor(graph.A, dtype=torch.float32).to(device)

# ===================== MODEL =====================
model = ASGCN_Model(
    num_class=3,
    num_point=25,
    num_person=9,
    in_channels=2
).to(device)

# ===================== LOSS + OPTIM =====================
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# ===================== DATALOADER =====================
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

# ===================== TRAIN =====================
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for x, y in tqdm(loader):
        x = x.float().to(device)
        y = y.to(device)

        out = model(x, A)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# ===================== EVALUATE =====================
def evaluate(model, loader):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in loader:

            x = x.float().to(device)
            y = y.to(device)

            out = model(x, A)

            pred = torch.argmax(out, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)

    prec = precision_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    rec = recall_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0
    )

    return acc, prec, rec, f1, y_true, y_pred


# ===================== TRAIN LOOP =====================
best_acc = 0

for epoch in range(200):
    print(f"\nEpoch {epoch+1}")

    train_loss = train_one_epoch(model, train_loader, optimizer)

    val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f} | Precision: {val_prec:.4f} | Recall: {val_rec:.4f} | F1: {val_f1:.4f}")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("✔ Saved best model")

    scheduler.step()


# ===================== TEST =====================
print("\nLoading best model...")
model.load_state_dict(torch.load("best_model.pth"))

test_acc, test_prec, test_rec, test_f1 = evaluate(model, test_loader)

print("\n===== TEST RESULT =====")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1-score : {test_f1:.4f}")